# Measuring Sensitivity

This notebook uses a local weak-signal capture or a synthetic fallback to explore the threshold at which recovered audio becomes unreliable. It frames sensitivity as a measurement process rather than a single spec-sheet number.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


In [ ]:
CAPTURE_ROOT = ROOT / "assets" / "local"
capture_status = probe_rtlsdr()
display(Markdown(
    f"**RTL-SDR status:** installed={capture_status['installed']}, "
    f"available={capture_status['available']}. {capture_status['message']}"
))


In [ ]:
capture_path = CAPTURE_ROOT / "sensitivity_test_iq.npz"
fs_iq = 96_000
t = np.arange(0, 2.0, 1 / fs_iq)
message = np.sin(2 * np.pi * 1000 * t)

if capture_path.exists():
    fs_iq, iq_clean = load_complex_capture(capture_path)
    print(f"Loaded local capture: {capture_path.name}, fs={fs_iq}")
else:
    iq_clean = synthesize_fm_iq(message, fs=fs_iq, carrier_offset=0, freq_dev=2000)
    print("Using synthetic weak-signal fallback.")


In [ ]:
audio_out = audio_output_widget()
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

def update_sensitivity(snr_db=20.0):
    noisy_i = add_awgn(iq_clean.real, snr_db=snr_db, seed=42)[0]
    noisy_q = add_awgn(iq_clean.imag, snr_db=snr_db, seed=43)[0]
    noisy_iq = noisy_i + 1j * noisy_q
    audio = fm_demodulate_iq(noisy_iq, fs=fs_iq, audio_cutoff=4000)
    tone_power = np.mean(message**2)
    err_power = np.mean((message[: len(audio)] - audio[: len(message)]) ** 2)
    snr_est = 10 * np.log10((tone_power + 1e-12) / (err_power + 1e-12))

    for ax in axes:
        ax.clear()
    plot_waveform(audio[:12000], fs=fs_iq, ax=axes[0], title="Recovered audio")
    plot_spectrum(audio, fs=fs_iq, ax=axes[1], title="Recovered audio spectrum")
    axes[1].set_xlim(0, 5000)
    axes[1].set_ylim(-100, 5)
    fig.canvas.draw_idle()
    with audio_out:
        audio_out.clear_output(wait=True)
        display(Markdown(f"**Injected RF SNR:** {snr_db:.1f} dB"))
        display(Markdown(f"**Estimated audio-domain SNR proxy:** {snr_est:.2f} dB"))
        display(audio_player(resample_signal(audio, fs_iq, 44_100), rate=44_100))

controls = widgets.interactive(
    update_sensitivity,
    snr_db=float_slider(min_value=-5, max_value=30, step=1, value=20, description="RF SNR"),
)
display(controls, audio_out)


## Key Takeaway

Sensitivity is a threshold problem. Whether you state it in dBm, SINAD, or recovered-audio quality, the workflow is the same: inject a weaker signal, measure the outcome, and define the fail point explicitly.